# Практика: Структурированный вывод с Groq API

В этом ноутбуке мы научимся получать от LLM не просто текст, а **структурированные данные** в формате JSON, которые можно сразу использовать в коде.

**Финальное задание:** построить мини-пайплайн для автоматического разбора резюме — программа будет извлекать из произвольного текста структурированный профиль кандидата.

## 0. Получение API-ключа Groq

1. Перейти на [developers.sber.ru](developers.sber.ru) и авторизоваться со Сбер ИД
2. Получить `Authorization Key`
3. Создать и заполнить `.env` с переменной `GIGACHAT_CREDENTIALS`

## 1. Установка и настройка клиента

In [3]:
!pip install gigachat python-dotenv

import os
import json
import time
from dotenv import load_dotenv
from gigachat import GigaChat
load_dotenv()

credentials = os.getenv("GIGACHAT_CREDENTIALS")

client = GigaChat(
    credentials=credentials,
    verify_ssl_certs=False
)

if credentials:
    print(f"Клиент GigaChat успешно инициализирован. Ключ: {credentials[:5]}...")
else:
    print("Ошибка: ключ GIGACHAT_CREDENTIALS не найден в файле .env")

Клиент GigaChat успешно инициализирован. Ключ: MDFhM...


## 2. Проблема: LLM возвращает неструктурированный текст

Попросим модель извлечь данные из текста обычным запросом — и посмотрим, почему такой вывод неудобен для программы.

In [4]:
resume_text = """
Привет! Меня зовут Анна Соколова, мне 28 лет. Последние 4 года работаю
бэкенд-разработчиком в финтех-компании. Основной стек: Python, FastAPI,
PostgreSQL, Redis. Есть опыт с Docker и Kubernetes. Закончила МГТУ им. Баумана
по специальности «Прикладная математика». Ищу позицию Senior Python Developer
с зарплатой от 300 000 рублей. Готова к удалённой работе или гибриду.
"""

from gigachat.models import Chat, Messages, MessagesRole

# Обычный запрос без требования структуры
chat_request = Chat(
    model="GigaChat-2",
    messages=[
        Messages(role=MessagesRole.USER, content=f"Извлеки основную информацию из резюме:\n{resume_text}")
    ]
)

# Отправляем запрос в GigaChat
response = client.chat(chat_request)

raw_output = response.choices[0].message.content
print('Вывод модели:')
print(raw_output)
print()

# Попытка использовать вывод в коде — неудача
print('Попытка получить зарплату из ответа:')
try:
    # Попытка превратить текст в словарь (JSON)
    data = json.loads(raw_output)
    print(data['salary'])
except Exception as e:
    print(f'Ошибка: {e}')
    print('Неструктурированный текст нельзя напрямую использовать в коде!')

Вывод модели:
### Основная информация из резюме:

**Имя:** Анна Соколова  
**Возраст:** 28 лет  
**Опыт работы:** 4 года бэкенд-разработчиком  
**Компания:** Финтех-компания  
**Ключевые технологии:**  
- Python  
- FastAPI  
- PostgreSQL  
- Redis  
- Docker  
- Kubernetes  
**Образование:**  
- МГТУ им. Баумана, специальность «Прикладная математика»  
**Цель:**  
- Поиск позиции Senior Python Developer  
- Желаемая зарплата: от 300 000 руб.  
- Готова к удаленной работе или гибридному формату.

Попытка получить зарплату из ответа:
Ошибка: Expecting value: line 1 column 1 (char 0)
Неструктурированный текст нельзя напрямую использовать в коде!


## 3. Решение: JSON-режим

~~Передадим параметр `response_format={'type': 'json_object'}` — и модель вернёт валидный JSON.~~

> ~~**Правило:** слово `JSON` должно присутствовать в системном промпте — иначе API вернёт ошибку.~~



In [5]:
resume_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "Имя кандидата"},
        "age": {"type": ["number", "null"], "description": "Возраст"},
        "experience_years": {"type": "number", "description": "Опыт работы в годах"},
        "current_position": {"type": "string", "description": "Текущая должность"},
        "skills": {"type": "array", "items": {"type": "string"}, "description": "Ключевые навыки"},
        "education": {"type": ["string", "null"], "description": "Образование"},
        "target_position": {"type": "string", "description": "Желаемая должность"},
        "salary_from": {"type": ["number", "null"], "description": "Ожидаемая зарплата"},
        "work_format": {"type": "string", "description": "Формат работы"}
    },
    "required": ["name", "experience_years", "skills", "target_position"]
}

In [6]:
system_prompt = """
Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго в формате JSON.
"""

chat_request = Chat(
    model="GigaChat-2",
    messages=[
        Messages(role=MessagesRole.SYSTEM, content=system_prompt),
        Messages(role=MessagesRole.USER, content=f'Обработай резюме:\n{resume_text}')
    ],
    response_format={
        "type": "json_schema",
        "schema": resume_schema,
        "strict": True
    }
)

response = client.chat(chat_request)

candidate = json.loads(response.choices[0].message.content)

print('Структурированный профиль кандидата:')
print(json.dumps(candidate, ensure_ascii=False, indent=2))
print()

# Теперь данные можно использовать в коде
print(f"Имя: {candidate['name']}")
print(f"Опыт: {candidate['experience_years']} лет")
print(f"Навыки: {', '.join(candidate['skills'])}")
print(f"Ожидаемая зарплата: {candidate['salary_from']:,} руб.".replace(',', ' '))

Структурированный профиль кандидата:
{
  "name": "Анна Соколова",
  "age": 28,
  "experience_years": 4,
  "current_position": "Бэкенд-разработчик",
  "skills": [
    "Python",
    "FastAPI",
    "PostgreSQL",
    "Redis",
    "Docker",
    "Kubernetes"
  ],
  "education": "МГТУ им. Баумана, специальность 'Прикладная математика'",
  "target_position": "Senior Python Developer",
  "salary_from": 300000,
  "work_format": "удалённая работа или гибрид"
}

Имя: Анна Соколова
Опыт: 4 лет
Навыки: Python, FastAPI, PostgreSQL, Redis, Docker, Kubernetes
Ожидаемая зарплата: 300 000 руб.


## 4. Пайплайн: обработка нескольких резюме

Завернём логику в функцию и запустим её на нескольких резюме сразу.

In [ ]:
import json
import time
from gigachat.models import Chat, Messages, MessagesRole
from gigachat.exceptions import RateLimitError

def parse_resume(text: str) -> dict | None:
    # Описание схемы
    resume_schema = {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "experience_years": {"type": "number"},
            "skills": {"type": "array", "items": {"type": "string"}},
            "target_position": {"type": "string"},
            "salary_from": {"type": ["number", "null"]},
            "work_format": {"type": "string"}
        },
        "required": ["name", "experience_years", "skills", "target_position", "salary_from", "work_format"]
    }
    
    system_prompt = "Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго JSON."
    
    try:
        chat_request = Chat(
            model="GigaChat-2",
            messages=[
                Messages(role=MessagesRole.SYSTEM, content=system_prompt),
                Messages(role=MessagesRole.USER, content=f'Обработай резюме:\n{text}')
            ],
            response_format={
                "type": "json_schema",
                "schema": resume_schema,
                "strict": True
            }
        )
        response = client.chat(chat_request)
        return json.loads(response.choices[0].message.content)
    except RateLimitError as e:
        print(f'Лимит запросов. Ждём 5 сек... ({e})')
        time.sleep(5)
        return None
    except Exception as e:
        print(f'Ошибка: {e}')
        return None

resumes = [
    """
    Иван Петров, 32 года. 7 лет в разработке игр на Unity и C#.
    Делал мобильные игры для iOS/Android, есть опыт с Blender.
    Ищу Lead Game Developer, готов к офису в Москве. Зарплата от 250 000.
    """,
    """
    Мария Иванова. Data Scientist с опытом 3 года.
    Стек: Python, pandas, scikit-learn, PyTorch, SQL.
    Работала в e-commerce над рекомендательными системами.
    Хочу ML Engineer, только удалёнка, зарплата не принципиальна.
    """,
    """
    Дмитрий Козлов, 25 лет, junior frontend. Знаю React, TypeScript, CSS.
    Полгода коммерческого опыта в веб-студии. Ищу работу в продуктовой компании.
    Рассматриваю от 120 000, гибрид или удалёнка.
    """,
    """
    Екатерина Смирнова, 29 лет. DevOps-инженер, 5 лет опыта.
    Умею: Kubernetes, Docker, Terraform, AWS, CI/CD (GitLab, GitHub Actions).
    Работала в банковском секторе, есть опыт с highload-системами.
    Ищу Senior DevOps, удалёнка обязательно. Зарплата от 300 000.
    """,
    """
    Алексей Новиков, 35 лет. Fullstack-разработчик, 10 лет опыта.
    Стек: Python (Django, FastAPI), Vue.js, PostgreSQL, Redis, Docker.
    Последние 3 года — тимлид команды из 6 человек в SaaS-компании.
    Рассматриваю роль Engineering Manager или Staff Engineer. От 400 000, гибрид.
    """,
    """
    Юлия Белова. QA Engineer, 4 года в тестировании.
    Автотесты на Python (pytest, Selenium, Playwright), API-тестирование (Postman, requests).
    Опыт в agile-командах, знакома с нагрузочным тестированием (Locust).
    Ищу Middle/Senior QA, гибрид, зарплата от 180 000.
    """,
    """
    Сергей Орлов, 27 лет. Android-разработчик, 3 года опыта.
    Kotlin, Jetpack Compose, Room, Retrofit, Coroutines. Публиковал 2 приложения в Google Play.
    Немного знаю iOS (Swift). Ищу Middle Android Developer в продуктовую компанию.
    Только удалёнка. Зарплата от 210 000.
    """
]

results = []
for i, resume in enumerate(resumes, 1):
    print(f'Обрабатываю резюме {i}/{len(resumes)}...')
    parsed = parse_resume(resume)
    if parsed:
        results.append(parsed)

print(f'\nОбработано: {len(results)} из {len(resumes)} резюме')
print()
for r in results:
    salary = f"{r['salary_from']:,}".replace(',', ' ') + ' руб.' if r.get('salary_from') else 'не указана'
    print(f"• {r['name']} | {r['target_position']} | опыт {r['experience_years']} л. | зарплата: {salary}")

Обрабатываю резюме 1/7...
Обрабатываю резюме 2/7...
Обрабатываю резюме 3/7...
Обрабатываю резюме 4/7...
Обрабатываю резюме 5/7...
Обрабатываю резюме 6/7...
Обрабатываю резюме 7/7...

Обработано: 7 из 7 резюме

• Иван Петров | Lead Game Developer | опыт 7 л. | зарплата: 250 000 руб.
• Мария Иванова | ML Engineer | опыт 3 л. | зарплата: не указана
• Дмитрий Козлов | junior frontend | опыт 0 л. | зарплата: 120 000 руб.
• Екатерина Смирнова | Senior DevOps | опыт 5 л. | зарплата: 300 000 руб.
• Алексей Новиков | Engineering Manager или Staff Engineer | опыт 10 л. | зарплата: 400 000 руб.
• Юлия Белова | Middle/Senior QA | опыт 4 л. | зарплата: 180 000 руб.
• Сергей Орлов | Middle Android Developer | опыт 3 л. | зарплата: 210 000 руб.


## 5. Фильтрация и анализ собранных данных

Структурированные данные позволяют легко фильтровать кандидатов без дополнительных запросов к модели.

Ниже показан один пример фильтрации — изучите его, затем **выполните самостоятельные задания** в следующей ячейке.


In [12]:
# Пример: кандидаты с опытом 3+ лет и зарплатой до 300 000
filtered = [
    r for r in results
    if r['experience_years'] >= 3
    and (r['salary_from'] is None or r['salary_from'] <= 300_000)
]

print(f'Кандидаты (опыт ≥ 3 лет, зарплата ≤ 300 000): {len(filtered)}')
for r in filtered:
    print(f"  → {r['name']}: {', '.join(r['skills'][:3])}...")

print()

# Самый опытный кандидат
if results:
    top = max(results, key=lambda r: r['experience_years'])
    print(f"Самый опытный: {top['name']} ({top['experience_years']} лет опыта)")

# Все упомянутые навыки
all_skills = [skill for r in results for skill in r['skills']]
unique_skills = sorted(set(all_skills))
print(f'\nВсе навыки в пуле: {len(unique_skills)}')
print(', '.join(unique_skills))


Кандидаты (опыт ≥ 3 лет, зарплата ≤ 300 000): 5
  → Иван Петров: Unity, C#, iOS development...
  → Мария Иванова: Python, pandas, scikit-learn...
  → Екатерина Смирнова: Kubernetes, Docker, Terraform...
  → Юлия Белова: Автотесты на Python (pytest, Selenium, Playwright), API-тестирование (Postman, requests), Опыт в agile-командах...
  → Сергей Орлов: Kotlin, Jetpack Compose, Room...

Самый опытный: Алексей Новиков (10 лет опыта)

Все навыки в пуле: 31
API-тестирование (Postman, requests), AWS, Android development, Blender, C#, CI/CD (GitLab, GitHub Actions), CSS, Coroutines, Docker, Jetpack Compose, Kotlin, Kubernetes, PostgreSQL, PyTorch, Python, Python (Django, FastAPI), React, Redis, Retrofit, Room, SQL, Terraform, TypeScript, Unity, Vue.js, iOS development, pandas, scikit-learn, Автотесты на Python (pytest, Selenium, Playwright), Знакома с нагрузочным тестированием (Locust), Опыт в agile-командах


### Задание: фильтрация кандидатов

Используя список `results`, выполните следующие задачи в ячейке ниже:

1. **Удалёнщики-синьоры** — найдите кандидатов, у которых `work_format` содержит слово `удалёнка` и `experience_years >= 4`.
2. **Бюджетный найм** — выведите всех кандидатов с зарплатными ожиданиями не выше 230 000 руб. (или без указанной зарплаты).
3. **Python-разработчики** — отберите кандидатов, у которых в `skills` есть `Python`.
4. **Топ-3 по опыту** — выведите трёх самых опытных кандидатов в порядке убывания стажа.
5. **Статистика зарплат** — посчитайте среднюю, минимальную и максимальную ожидаемую зарплату среди тех, кто её указал.


In [13]:
# Задание 1: удалёнщики с опытом 4+ лет
# TODO: напишите list comprehension или цикл


# Задание 2: зарплатные ожидания <= 230 000 (или не указаны)
# TODO:


# Задание 3: кандидаты с Python в навыках
# Подсказка: используйте any() или проверку 'Python' in r['skills']
# TODO:


# Задание 4: топ-3 по опыту
# Подсказка: sorted(..., key=..., reverse=True)[:3]
# TODO:


# Задание 5: статистика зарплат
# Подсказка: сначала соберите список salary_from тех, у кого он не None
# TODO:


Реализация фильтров по заданию на основании примеров выше:

In [14]:
print("--- Задание 1: Удалёнщики-синьоры ---")
# Поиск 'удалён' в любом регистре и опыта от 4 лет
remote_seniors = [
    r for r in results 
    if 'удалён' in str(r.get('work_format', '')).lower() and r['experience_years'] >= 4
]
for r in remote_seniors:
    print(f"• {r['name']} | Опыт: {r['experience_years']} лет | Формат: {r['work_format']}")


print("\n--- Задание 2: Бюджетный найм (до 230 000 или без зп) ---")
budget_candidates = [
    r for r in results 
    if r.get('salary_from') is None or r['salary_from'] <= 230_000
]
for r in budget_candidates:
    salary = f"{r['salary_from']:,} руб.".replace(',', ' ') if r.get('salary_from') else 'не указана'
    print(f"• {r['name']} | Зарплата: {salary}")


print("\n--- Задание 3: Python-разработчики ---")
# Проверка наличия 'python' в любом регистре внутри списка навыков
python_devs = [
    r for r in results 
    if any('python' in skill.lower() for skill in r.get('skills', []))
]
for r in python_devs:
    print(f"• {r['name']} | Навыки: {', '.join(r['skills'])}")


print("\n--- Задание 4: Топ-3 по опыту ---")
# Сортировка по опыту (по убыванию) и выбор первых 3
top_3 = sorted(results, key=lambda x: x['experience_years'], reverse=True)[:3]
for i, r in enumerate(top_3, 1):
    print(f"{i}. {r['name']} — {r['experience_years']} лет опыта")


print("\n--- Задание 5: Статистика зарплат ---")
salaries = [r['salary_from'] for r in results if r.get('salary_from') is not None]
if salaries:
    avg_salary = sum(salaries) / len(salaries)
    print(f"Кандидатов с указанной зарплатой: {len(salaries)}")
    print(f"Минимальная зарплата: {min(salaries):,} руб.".replace(',', ' '))
    print(f"Максимальная зарплата: {max(salaries):,} руб.".replace(',', ' '))
    print(f"Средняя зарплата: {avg_salary:,.0f} руб.".replace(',', ' '))
else:
    print("Никто из кандидатов не указал зарплату.")

--- Задание 1: Удалёнщики-синьоры ---
• Екатерина Смирнова | Опыт: 5 лет | Формат: удалёнка

--- Задание 2: Бюджетный найм (до 230 000 или без зп) ---
• Мария Иванова | Зарплата: не указана
• Дмитрий Козлов | Зарплата: 120 000 руб.
• Юлия Белова | Зарплата: 180 000 руб.
• Сергей Орлов | Зарплата: 210 000 руб.

--- Задание 3: Python-разработчики ---
• Мария Иванова | Навыки: Python, pandas, scikit-learn, PyTorch, SQL
• Алексей Новиков | Навыки: Python (Django, FastAPI), Vue.js, PostgreSQL, Redis, Docker
• Юлия Белова | Навыки: Автотесты на Python (pytest, Selenium, Playwright), API-тестирование (Postman, requests), Опыт в agile-командах, Знакома с нагрузочным тестированием (Locust)

--- Задание 4: Топ-3 по опыту ---
1. Алексей Новиков — 10 лет опыта
2. Иван Петров — 7 лет опыта
3. Екатерина Смирнова — 5 лет опыта

--- Задание 5: Статистика зарплат ---
Кандидатов с указанной зарплатой: 6
Минимальная зарплата: 120 000 руб.
Максимальная зарплата: 400 000 руб.
Средняя зарплата: 243 333 руб.

## 6. Сравнение моделей

Groq предоставляет доступ к нескольким моделям с разными характеристиками. Сравним их по **скорости** и **качеству** извлечения структурированных данных.

| Модель | Размер | Типичная скорость |
|---|---|---|
| `llama-3.1-8b-instant` | 8B | ~1000–1500 токенов/с |
| `llama-3.3-70b-versatile` | 70B | ~250–400 токенов/с |
| `gemma2-9b-it` | 9B | ~500–900 токенов/с |
| `mixtral-8x7b-32768` | 8×7B MoE | ~400–700 токенов/с |

> 💡 Актуальный список моделей всегда доступен на [console.groq.com/docs/models](https://console.groq.com/docs/models)


In [15]:
import time
import json
from gigachat.models import Chat, Messages, MessagesRole

# Резюме для теста
test_resume = """
Андрей Захаров, 31 год. Backend-разработчик, 6 лет опыта.
Основной стек: Go, Python, gRPC, Kafka, PostgreSQL, Kubernetes.
Строил высоконагруженные сервисы в финтех и логистике.
Ищу Senior/Lead Backend Engineer. Удалёнка, зарплата от 380 000.
"""

# Схема для строгого JSON
test_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "experience_years": {"type": "number"},
        "skills": {"type": "array", "items": {"type": "string"}},
        "target_position": {"type": "string"},
        "salary_from": {"type": ["number", "null"]},
        "work_format": {"type": "string"}
    },
    "required": ["name", "experience_years", "skills", "target_position"]
}

system_prompt = "Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго JSON."

# Изменение списка моделей на актуальные для GigaChat
models_to_compare = [
    "GigaChat-2",       # Lite
    "GigaChat-2-Pro",   # Pro
    "GigaChat-2-Max",   # Max
]

comparison_results = []

for model_name in models_to_compare:
    try:
        start = time.time()
        chat_request = Chat(
            model=model_name,
            messages=[
                Messages(role=MessagesRole.SYSTEM, content=system_prompt),
                Messages(role=MessagesRole.USER, content=f'Обработай резюме:\n{test_resume}')
            ],
            response_format={
                "type": "json_schema",
                "schema": test_schema,
                "strict": True
            }
        )
        response = client.chat(chat_request)
        elapsed = time.time() - start

        parsed = json.loads(response.choices[0].message.content)
        usage = response.usage

        comparison_results.append({
            'model': model_name,
            'time_sec': round(elapsed, 2),
            'total_tokens': usage.total_tokens,
            'tokens_per_sec': round(usage.completion_tokens / elapsed) if elapsed > 0 else 0,
            'skills_found': len(parsed.get('skills', [])),
            'parsed': parsed,
        })
        print(f'OK {model_name}: {elapsed:.2f} с, {usage.total_tokens} токенов')
        time.sleep(1)  # пауза между запросами
    except Exception as e:
        print(f'ОШИБКА {model_name}: {e}')

print()
header = f"{'Модель':<20} {'Время':>8} {'Ток/с':>8} {'Навыков':>9}"
print(header)
print('-' * 50)
for r in comparison_results:
    row = f"{r['model']:<20} {r['time_sec']:>7.2f}с {r['tokens_per_sec']:>8} {r['skills_found']:>9}"
    print(row)

OK GigaChat-2: 0.99 с, 238 токенов
OK GigaChat-2-Pro: 1.45 с, 222 токенов
OK GigaChat-2-Max: 1.93 с, 204 токенов

Модель                  Время    Ток/с   Навыков
--------------------------------------------------
GigaChat-2              0.99с      116         6
GigaChat-2-Pro          1.45с       68         6
GigaChat-2-Max          1.93с       45         6


### Задание: анализ результатов сравнения

После выполнения ячейки выше ответьте на вопросы (в новой ячейке типа Markdown или в комментариях):

1. Какая модель оказалась **самой быстрой** по времени ответа? А по числу токенов в секунду?
2. Все ли модели извлекли одинаковое количество навыков? Есть ли расхождения?
3. Когда стоит предпочесть большую модель (70B) маленькой (8B)? Приведите пример задачи.
4. **Бонус:** добавьте в список ещё одну модель из документации Groq и запустите сравнение заново.


**Анализ результатов сравнения моделей:**

1. **Самая быстрая:** По времени и по количеству токенов в секунду лидирует `GigaChat-2` (Lite). Как говорится "для народа".
2. **Количество навыков:** Все модели нашли одинаковое количество навыков (6). Расхождений на тесте нет.
3. **Когда выбирать большую модель (Max):** Модель `GigaChat-2-Max` стоит использовать, когда требуется глубокое понимание сложного контекста (например, программирование, анализ длинных текстов) или когда цена ошибки высока (например, финансы). Модель `GigaChat-2` (Lite) лучше подходит для простых задач, где важна скорость и экономия токенов.

---
## Самостоятельное задание: парсер вакансий

Вы видели, как разбирать **резюме**. Теперь сделайте то же самое для **вакансий**.

### Задача
Реализуйте функцию `parse_vacancy(text: str) -> dict | None`, которая принимает текст вакансии и возвращает структурированный JSON со следующими полями:

```json
{
  "position": "название должности",
  "company": "название компании или null",
  "required_skills": ["список обязательных навыков"],
  "nice_to_have": ["список желательных навыков"],
  "experience_required": "например '3-5 лет' или null",
  "salary_to": "верхняя граница зарплаты или null",
  "work_format": "офис / удалёнка / гибрид"
}
```

### Требования
- Используйте `response_format={'type': 'json_object'}`
- Системный промпт должен содержать слово `JSON` и описание схемы
- Оберните вызов в `try/except`
- Проверьте функцию на всех пяти тестовых вакансиях ниже

### Бонус ⭐
Напишите функцию `match_candidates(vacancy: dict, candidates: list) -> list`, которая по уже разобранным данным подбирает подходящих кандидатов из списка `results` (совпадение хотя бы 2 навыков из `required_skills`).

In [19]:
# Тестовые вакансии
vacancy_1 = """
Компания TechFlow ищет Senior Python Developer.
Обязательно: Python 5+ лет, FastAPI или Django, PostgreSQL, Docker.
Желательно: Kubernetes, опыт с микросервисами, Redis.
Условия: гибрид (Москва), зарплата до 400 000 рублей, ДМС.
"""

vacancy_2 = """
Ищем Middle Data Scientist в e-commerce стартап.
Требования: Python, pandas, scikit-learn, опыт построения ML-моделей от 2 лет.
Плюсом будет: PyTorch, опыт с рекомендательными системами, A/B-тесты.
Полная удалёнка, зарплата до 220 000. Команда 5 человек.
"""

vacancy_3 = """
Frontend Developer (React) — продуктовая компания в сфере EdTech.
Must have: React, TypeScript, REST API, опыт от 1 года.
Nice to have: Next.js, тестирование (Jest), Storybook.
Офис в СПб или удалёнка, вилка 150 000 – 200 000.
"""


def parse_vacancy(text: str) -> dict | None:
    """
    Извлекает структурированные данные из текста вакансии.
    """
    # Схема для извлечения данных вакансии
    vacancy_schema = {
        "type": "object",
        "properties": {
            "position": {"type": "string"},
            "company": {"type": ["string", "null"]},
            "required_skills": {"type": "array", "items": {"type": "string"}},
            "nice_to_have": {"type": "array", "items": {"type": "string"}},
            "experience_required": {"type": ["string", "null"]},
            "salary_to": {"type": ["number", "null"]},
            "work_format": {"type": "string"}
        },
        "required": ["position", "company", "required_skills", "nice_to_have", "experience_required", "salary_to", "work_format"]
    }
    
    system_prompt = "Ты — HR-ассистент. Извлекай данные из вакансии и возвращай строго JSON."
    
    try:
        chat_request = Chat(
            model="GigaChat-2",  # Быстрая модель
            messages=[
                Messages(role=MessagesRole.SYSTEM, content=system_prompt),
                Messages(role=MessagesRole.USER, content=f'Обработай вакансию:\n{text}')
            ],
            response_format={
                "type": "json_schema",
                "schema": vacancy_schema,
                "strict": True
            }
        )
        response = client.chat(chat_request)
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f'Ошибка при парсинге вакансии: {e}')
        return None


# Тестирование
for i, vacancy in enumerate([vacancy_1, vacancy_2, vacancy_3], 1):
    print(f'{"=" * 55}')
    print(f'Вакансия {i}')
    print('=' * 55)
    result = parse_vacancy(vacancy)
    if result:
        print(json.dumps(result, ensure_ascii=False, indent=2))
    print()

Вакансия 1
{
  "position": "Senior Python Developer",
  "company": "TechFlow",
  "required_skills": [
    "Python (5+ лет)",
    "FastAPI или Django",
    "PostgreSQL",
    "Docker"
  ],
  "nice_to_have": [
    "Kubernetes",
    "опыт с микросервисами",
    "Redis"
  ],
  "experience_required": "гибрид (Москва)",
  "salary_to": 400000,
  "work_format": "гибрид (Москва)"
}

Вакансия 2
{
  "position": "Middle Data Scientist",
  "company": "e-commerce стартап",
  "required_skills": [
    "Python",
    "pandas",
    "scikit-learn",
    "опыт построения ML-моделей от 2 лет"
  ],
  "nice_to_have": [
    "PyTorch",
    "опыт с рекомендательными системами",
    "A/B-тесты"
  ],
  "experience_required": "от 2 лет",
  "salary_to": 220000,
  "work_format": "полная удалёнка"
}

Вакансия 3
{
  "position": "Frontend Developer (React)",
  "company": "продуктовая компания в сфере EdTech",
  "required_skills": [
    "React",
    "TypeScript",
    "REST API",
    "опыт от 1 года"
  ],
  "nice_to_have": 

In [20]:
# Бонус: подбор кандидатов под вакансию

def match_candidates(vacancy: dict, candidates: list) -> list:
    """
    Подбирает кандидатов из списка по совпадению навыков.

    Args:
        vacancy: Разобранная вакансия (результат parse_vacancy)
        candidates: Список разобранных резюме (результаты parse_resume)

    Returns:
        Список подходящих кандидатов с указанием совпавших навыков
    """
    # 1. Возьмите required_skills из vacancy
    required_skills = {skill.lower() for skill in vacancy.get('required_skills', [])}
    matched_candidates = []

    # 2. Для каждого кандидата найдите пересечение его skills с required_skills
    for candidate in candidates:
        candidate_skills = {skill.lower() for skill in candidate.get('skills', [])}
        common_skills = required_skills.intersection(candidate_skills)

        # 3. Верните кандидатов, у которых совпало >= 2 навыков
        if len(common_skills) >= 2:
            # 4. В возвращаемый словарь добавьте поле matched_skills
            candidate_copy = candidate.copy()
            candidate_copy['matched_skills'] = list(common_skills)
            matched_candidates.append(candidate_copy)

    return matched_candidates

# Проверка работы (можно добавить в конец этой же ячейки)
print("--- Подходящие кандидаты на вакансию 1 ---")
for m in match_candidates(parse_vacancy(vacancy_1), results):
    print(f"{m['name']} | {', '.join(m['matched_skills'])}")

--- Подходящие кандидаты на вакансию 1 ---
Алексей Новиков | postgresql, docker


---
## Чек-лист перед сдачей

- [ ] Все ячейки выполняются без ошибок (Kernel → Restart & Run All)
- [ ] Раздел 5: выполнены все 5 заданий по фильтрации
- [ ] Раздел 6: таблица сравнения моделей заполнена, письменные ответы добавлены
- [ ] Функция `parse_vacancy()` реализована и возвращает корректный JSON для всех 5 вакансий
- [ ] Все 7 полей схемы присутствуют в ответе (или `null`, если данных нет)
- [ ] Используется `response_format={'type': 'json_object'}`
- [ ] В коде есть обработка ошибок `try/except`
- [ ] API-ключ не хранится в коде — только через `.env`
- [ ] (Бонус) Функция `match_candidates()` реализована
